# Testing Spark Pipeline to PMML

Objective: To test pipeline (data processing + spark xgboost) and convert to PMML.

### Packages/Libraries Installation

In [1]:
!java --version

openjdk 17.0.17 2025-10-21
OpenJDK Runtime Environment (build 17.0.17+10-Ubuntu-122.04)
OpenJDK 64-Bit Server VM (build 17.0.17+10-Ubuntu-122.04, mixed mode, sharing)


In [2]:
!python --version

Python 3.12.12


In [3]:
!pip install pyspark==3.5.0
!pip uninstall -y dataproc-spark-connect # can comment this out, if the pyspark version required is correctly installed

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.9/316.9 MB 4.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 15.3 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-3.5.0-py2.py3-none-any.whl size=317425346 sha256=da85c446637e707820a18eff680238e719d804db26ad15bebdabe5134d3e91d9
  Stored in directory: /root/.cache/pip/wheels/84/40/20/65eefe766118e0a8f8e385cc3ed6e9eb7241c7e51cfc04c51a
Successfully built pyspark
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.9
    Uninstalling py4j-0.10.9.9:
      Successfully uninstalled py4j-0.10.9.9
  Attempting uninstall: pyspark
    Found existing installation: pyspark 4.0.1
    Uninstalling pyspark-4.0.1:
      Successfully uninstalled pyspark-4.0.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-conn

In [59]:
import os
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.ml.feature import (
    StringIndexer,
    OneHotEncoder,
    Imputer,
    VectorAssembler,
)
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml.wrapper import JavaEstimator, JavaModel
from pyspark.ml.util import JavaMLWritable, JavaMLReadable

# only need these to create test samples
from pyspark.sql.types import StructType, StructField, DoubleType, StringType, IntegerType

### Environment Setup

Configure the Spark environment by setting the PYSPARK_SUBMIT_ARGS environment variable to include the required JPMML and XGBoost libraries.


In [6]:
# must run in fresh kernel, before creating spark session
packages = ",".join([
    "org.jpmml:pmml-sparkml:3.1.9",         #generic PMML exporter for Spark ML
    "org.jpmml:pmml-sparkml-xgboost:3.1.9", #bridges Spark XGBoost models → PMML
    "ml.dmlc:xgboost4j-spark_2.12:3.1.1",   #implement/train xgb
])

os.environ["PYSPARK_SUBMIT_ARGS"] = f'--packages {packages} pyspark-shell'


In [60]:
spark = (SparkSession.builder
         .appName("XGBoostPMML")
         .getOrCreate())

print("Spark:", spark.version)
print("SparkSession initialized with XGBoost and PMML packages.")


Spark: 3.5.0
SparkSession initialized with XGBoost and PMML packages.


### Create Python Wrapper for Scala-based XGB

Python wrapper that allows PySpark to use the Scala-based XGBoost classifier by forwarding parameters from Python to the underlying Java/Scala implementation

In [8]:
# Wrapper for the XGBoost Model
class XGBoostClassificationModel(JavaModel):
    pass

# Wrapper for the XGBoost Classifier Estimator
class XGBoostClassifier(JavaEstimator, JavaMLWritable, JavaMLReadable):
    def __init__(self, **kwargs):
        super(XGBoostClassifier, self).__init__()
        # Instantiate the Scala XGBoostClassifier object
        self._java_obj = self._new_java_obj("ml.dmlc.xgboost4j.scala.spark.XGBoostClassifier")

        # Iterate through kwargs and call the corresponding setters on the Java object
        for param, value in kwargs.items():
            # Construct the setter name (e.g., 'objective' -> 'setObjective')
            setter_name = "set" + param[0].upper() + param[1:]
            if hasattr(self._java_obj, setter_name):
                getattr(self._java_obj, setter_name)(value)
            else:
                print(f"Warning: Setter {setter_name} not found on Java object.")

    def _create_model(self, java_model):
        # Return the Python wrapper for the created Java model
        return XGBoostClassificationModel(java_model)

### Create Samples Dataset

In [53]:
data = [
    (1.2, "cat_a", 5.6, 0),
    (4.5, "cat_b", 8.9, 1),
    (2.1, "cat_a", None, 0), # Missing f3
    (5.4, "cat_b", 9.8, 1),
    (1.0, "cat_a", 3.0, 0),
    (None, "cat_b", 6.0, 1), # Missing f1
    (2.2, "cat_a", 4.4, 0),
    (5.5, "cat_b", 7.7, 1)
]

# Define schema to ensure correct types (especially for None values)
schema = StructType([
    StructField("f1", DoubleType(), True),
    StructField("f2", StringType(), True), # Categorical
    StructField("f3", DoubleType(), True),
    StructField("label", IntegerType(), True)
])
df = spark.createDataFrame(data, schema)


Pipeline with Imputation and OHE created.
+---+-----+----+-----+
| f1|   f2|  f3|label|
+---+-----+----+-----+
|1.2|cat_a| 5.6|    0|
|4.5|cat_b| 8.9|    1|
|2.1|cat_a|NULL|    0|
|5.4|cat_b| 9.8|    1|
|1.0|cat_a| 3.0|    0|
+---+-----+----+-----+
only showing top 5 rows

Fitting pipeline...

Predictions:
+---+-----+----+----------+---------------------------------------+
|f1 |f2   |f3  |prediction|probability                            |
+---+-----+----+----------+---------------------------------------+
|1.2|cat_a|5.6 |0.0       |[0.5744425058364868,0.4255574941635132]|
|4.5|cat_b|8.9 |1.0       |[0.4255574941635132,0.5744425058364868]|
|2.1|cat_a|NULL|0.0       |[0.5744425058364868,0.4255574941635132]|
|5.4|cat_b|9.8 |1.0       |[0.4255574941635132,0.5744425058364868]|
|1.0|cat_a|3.0 |0.0       |[0.5744425058364868,0.4255574941635132]|
+---+-----+----+----------+---------------------------------------+
only showing top 5 rows



### Pipeline (preprocessing + assembler + xgb)

In [63]:

imputer = Imputer(inputCols=["f1", "f3"], outputCols=["f1_imp", "f3_imp"], strategy="mean")
indexer = StringIndexer(inputCol="f2", outputCol="f2_idx", handleInvalid="keep")
encoder = OneHotEncoder(inputCols=["f2_idx"], outputCols=["f2_ohe"])

# Assembling into a single vector
assembler = VectorAssembler(inputCols=["f1_imp", "f2_ohe", "f3_imp"], outputCol="features")

# XGB Classifier
xgb = XGBoostClassifier(labelCol="label", featuresCol="features", objective="binary:logistic", numWorkers=1, nthread=1)

pipeline = Pipeline(stages=[imputer, indexer, encoder, assembler, xgb])
print("Pipeline with Imputation and OHE created.")
df.show(5)

Pipeline with Imputation and OHE created.
+---+-----+----+-----+
| f1|   f2|  f3|label|
+---+-----+----+-----+
|1.2|cat_a| 5.6|    0|
|4.5|cat_b| 8.9|    1|
|2.1|cat_a|NULL|    0|
|5.4|cat_b| 9.8|    1|
|1.0|cat_a| 3.0|    0|
+---+-----+----+-----+
only showing top 5 rows



In [65]:
print("Fitting pipeline...")
pipeline_model = pipeline.fit(df)
preds = pipeline_model.transform(df)

print("\nPredictions:")
preds.select("f1", "f2", "f3", "prediction", "probability").show(5, truncate=False)

Fitting pipeline...

Predictions:
+---+-----+----+----------+---------------------------------------+
|f1 |f2   |f3  |prediction|probability                            |
+---+-----+----+----------+---------------------------------------+
|1.2|cat_a|5.6 |0.0       |[0.5744425058364868,0.4255574941635132]|
|4.5|cat_b|8.9 |1.0       |[0.4255574941635132,0.5744425058364868]|
|2.1|cat_a|NULL|0.0       |[0.5744425058364868,0.4255574941635132]|
|5.4|cat_b|9.8 |1.0       |[0.4255574941635132,0.5744425058364868]|
|1.0|cat_a|3.0 |0.0       |[0.5744425058364868,0.4255574941635132]|
+---+-----+----+----------+---------------------------------------+
only showing top 5 rows



### Convert XGB to PMML

In [67]:
# Access the JVM gateway directly (alternative to installing pyspark2pmml) and convert Python objects to Java objects
java_pipeline_model = pipeline_model._to_java()
java_schema = df._jdf.schema()

# Define PMMLBuilder from the loaded JARs
PMMLBuilder = spark.sparkContext._jvm.org.jpmml.sparkml.PMMLBuilder

# Instantiate the builder with the schema and model
pmml_builder = PMMLBuilder(java_schema, java_pipeline_model)
print("PMMLBuilder instantiated successfully via JVM.")

JavaFile = spark.sparkContext._jvm.java.io.File
target_file = JavaFile("xgboost_model.pmml")

# Export the model using the pmml_builder instance
pmml_builder.buildFile(target_file)

if os.path.exists("xgboost_model.pmml"):
    print("PMML file exported successfully to xgboost_model.pmml")
else:
    print("Failed to export PMML file.")

PMMLBuilder instantiated successfully via JVM.
PMML file exported successfully to xgboost_model.pmml


### Test the Model, not the PMML

In [68]:
test_data = [
    (3.2, "cat_a", 6.7),
    (4.5, "cat_c", 7.0),
    (4.5, "cat_a", None),
]
test_df = spark.createDataFrame(test_data, ["f1", "f2", "f3"])

# Use the trained pipeline model for inference
predictions = pipeline_model.transform(test_df)
print("Inference Results:")
predictions.select("f1", "f2", "f3", "prediction").show()

Inference Results:
+---+-----+----+----------+
| f1|   f2|  f3|prediction|
+---+-----+----+----------+
|3.2|cat_a| 6.7|       1.0|
|4.5|cat_c| 7.0|       1.0|
|4.5|cat_a|NULL|       1.0|
+---+-----+----+----------+



### Load and Test PMML Model

In [75]:
import os
import pandas as pd
from pyspark.sql import SparkSession

# stop current spark
try:
    spark.stop()
except NameError:
    pass

if "PYSPARK_SUBMIT_ARGS" in os.environ:
    del os.environ["PYSPARK_SUBMIT_ARGS"]

# init Spark with the pmml4s-spark package
spark = SparkSession.builder \
    .appName("PMMLScoring") \
    .config("spark.jars.packages", "org.pmml4s:pmml4s-spark_2.12:1.0.0") \
    .getOrCreate()

# using same test data
test_data = [
    (3.2, "cat_a", 6.7),
    (4.5, "cat_c", 7.0),
    (4.5, "cat_a", None),
]
test_df = spark.createDataFrame(test_data, ["f1", "f2", "f3"])
print("Spark Configured Packages:", spark.sparkContext.getConf().get("spark.jars.packages"))

try:
    # TODO: Need to fix this to test using pypmml-spark/Java based
    from pypmml_spark import ScoreModel
    model = ScoreModel.fromFile('xgboost_model.pmml')
    score_df = model.transform(test_df)
    print("Successfully using pypmml-spark.")


except Exception as e:
    print(f"\nWarning: pypmml-spark failed ({e}). Falling back to pypmml (Python-based)...")
    from pypmml import Model
    model = Model.load('xgboost_model.pmml')
    test_pd = test_df.toPandas()
    predictions = model.predict(test_pd)
    result_pd = pd.concat([test_pd, predictions], axis=1)
    score_df = spark.createDataFrame(result_pd)
    print("Successfully using pypmml used (via Pandas conversion).")

print("PMML Inference Results (Spark DataFrame):")
score_df.select("f1", "f2", "f3", "prediction").show()

Spark Configured Packages: org.pmml4s:pmml4s-spark_2.12:1.0.0
Attempting to use pypmml-spark (Java-based)...

Success: pypmml used (via Pandas conversion).
PMML Inference Results (Spark DataFrame):
+---+-----+---+----------+
| f1|   f2| f3|prediction|
+---+-----+---+----------+
|3.2|cat_a|6.7|         1|
|4.5|cat_c|7.0|         1|
|4.5|cat_a|NaN|         1|
+---+-----+---+----------+

